# Smoke page: loading one FRESH recording with Cedalion

[Open this notebook in Google Colab](https://colab.research.google.com/github/Ramdam17/fnirs-tutorials/blob/main/book/smoke/smoke_cedalion.ipynb)

This is a smoke page: it exists to test the build machinery, it is not a chapter, and it is removed from the table of contents before version 1.0.

## Objectives, prerequisites and reading time

- Objective: check that the book builds, that a notebook executes headless, and that Cedalion reads a FRESH recording through the path helper of the project.
- Prerequisites: none; the page teaches no fNIRS concept.
- Estimated reading time: one minute.

## Concept

This section is a placeholder that exercises the second part of the chapter template. A real chapter states one concept here, with one or two figures.

## Hands-on

The next cells read one motor recording of the FRESH dataset, report its size and sampling rate, and plot two seconds of raw intensity on one short and one long channel.

In [ ]:
# @title Setup for Google Colab
# Anywhere else this cell does nothing. On Colab it installs the environment and the data
# of the book once, then restarts the session on purpose. After the restart, run the
# notebook again from the top. The first line is the title Colab shows above this cell,
# which it displays folded behind "Show code".
# Generated by scripts/sync_colab_cells.py from book/_templates/colab_setup.py: edit the
# template, not this cell.
import importlib as _importlib
import importlib.metadata as _metadata
import json as _json
import os as _os
import shutil as _shutil
import subprocess as _subprocess
import sys as _sys
import time as _time
import urllib.request as _request
from pathlib import Path as _Path

# --- values written by scripts/sync_colab_cells.py (begin) ---
_REPOSITORY = "Ramdam17/fnirs-tutorials"
_REF = "main"
_OVERRIDES = ["scipy==1.16.3", "statsmodels==0.14.6"]
_RUNTIME_VERSION = "2026.07"
_PYTHON_VERSION = "3.12"
_DATA_DIR = "/content/fnirs-data"
_CACHE_DIR = "/content/fnirs-cache"
_SETUP_DIR = "/content/fnirs-setup"
# --- values written by scripts/sync_colab_cells.py (end) ---

# The line that reports the duration of the setup starts with this prefix, so that it can
# be found in the output of the cell. Colab shows no execution time for a cell whose
# session ended, which is why the cell prints its own.
_TIMES_PREFIX = "Setup time in seconds:"
_MARKER_NAME = "installed.json"
_OVERRIDES_NAME = "overrides.txt"
_DOWNLOAD_SCRIPT_NAME = "download_data.py"
# Seconds to wait between the announcement and the end of the session, so that the text
# reaches the browser first: measured on Colab, a line printed just before the kill was
# lost, and the same line printed with a flush and followed by a wait was shown. Two
# seconds is the value of Cedalion's Colab script, the only one tried: a heuristic.
_FLUSH_WAIT_S = 2
# Shared by every step that can fail, so that a newcomer always reads the same advice.
_ADVICE = (
    'Run this cell again. If it fails a second time, open "Runtime", '
    '"Disconnect and delete runtime", then run the notebook again. If it still '
    "fails, report the problem with the lines printed above."
)
# Printed before the install, because uv prints nothing while it downloads the packages.
# Measured on Colab on two runs, the install took 55 s and 417 s, and the second one
# showed no new line for more than six minutes. The bounds of the sentence frame these
# two runs with a margin; they are not a measured range: a heuristic.
_INSTALL_NOTICE = (
    "Installing the environment of the book. This takes from one to ten minutes, and "
    "the install can print nothing for several minutes. That is expected: let it run."
)
_ANNOUNCEMENT = (
    "The environment is installed.\n"
    "The session now restarts once, on purpose, so that the new packages are loaded.\n"
    'Colab will show the notice "Your session crashed for an unknown reason". '
    "That is expected here.\n"
    "When the session is back, run the notebook again from the top with "
    '"Runtime", "Run all". This cell will then take about a second.'
)


def _on_colab() -> bool:
    # The one test for "this is Colab": Colab's kernel provides the google.colab module.
    return "google.colab" in _sys.modules


def _running_python() -> str:
    return f"{_sys.version_info.major}.{_sys.version_info.minor}"


def _run(command: list[str]) -> None:
    # The output is read line by line and printed here: a process started from a kernel
    # does not reliably show its own output in the cell. No shell is involved.
    print("$ " + " ".join(command), flush=True)
    process = _subprocess.Popen(
        command,
        stdout=_subprocess.PIPE,
        stderr=_subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env={**_os.environ, "PYTHONUNBUFFERED": "1"},
    )
    assert process.stdout is not None
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    except BaseException:
        # An interruption (the Stop button included) must not leave the install running
        # beside the next one: end the child, wait for it, then raise again.
        process.kill()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if code != 0:
        raise RuntimeError(f"The command {command} ended with exit code {code}.")


def _fetch(url: str, destination: _Path) -> None:
    print(f"Fetching {url}", flush=True)
    with _request.urlopen(url, timeout=60) as response:
        destination.write_bytes(response.read())


def _kill_kernel() -> None:
    # Ending the kernel process is how Cedalion's own Colab script restarts the session:
    # Colab starts a new kernel by itself.
    _os.kill(_os.getpid(), 9)


def _end_session() -> None:
    print(_ANNOUNCEMENT, flush=True)
    _sys.stdout.flush()
    _sys.stderr.flush()
    _time.sleep(_FLUSH_WAIT_S)
    _kill_kernel()


def _marker_matches(marker: _Path) -> bool:
    expected = {"ref": _REF, "overrides": _OVERRIDES}
    try:
        return _json.loads(marker.read_text(encoding="utf-8")) == expected
    except (OSError, ValueError):
        return False


def _set_environment() -> None:
    _os.environ["FNIRS_DATA_DIR"] = _DATA_DIR
    _os.environ["FNIRS_CACHE_DIR"] = _CACHE_DIR


def _check_installed() -> None:
    # `uv pip install --system` installs into the first Python it finds, which should be
    # the one of this session. Checking before the marker is written keeps a wrong
    # install from being reported as ready on every later run.
    _importlib.invalidate_caches()
    try:
        _metadata.version("fnirs-tutorials")
    except _metadata.PackageNotFoundError as error:
        raise RuntimeError(
            "The install finished, but the Python of this session does not find the "
            "package fnirs-tutorials: it was installed for another Python."
        ) from error


def _kernel_has_old_numpy() -> bool:
    # The kernel imports numpy before the first cell. If the install replaced it on disk,
    # the kernel still holds the old one, and importing Cedalion then fails.
    loaded = _sys.modules.get("numpy")
    return loaded is not None and loaded.__version__ != _metadata.version("numpy")


def _setup() -> None:
    running = _running_python()
    if running != _PYTHON_VERSION:
        raise RuntimeError(
            f"This notebook needs Python {_PYTHON_VERSION}, and this Colab session runs "
            f'Python {running}. To change it, open "Runtime", then "Change runtime '
            f'type", choose the version {_RUNTIME_VERSION} in the list "Runtime version", '
            "and run this cell again."
        )

    setup_dir = _Path(_SETUP_DIR)
    marker = setup_dir / _MARKER_NAME
    if _marker_matches(marker):
        # A restart loses the environment of the kernel, and a non-editable install has
        # no repository above the package to find these folders from.
        _set_environment()
        if _kernel_has_old_numpy():
            _end_session()
        else:
            print("The environment is ready.")
        return

    if _shutil.which("uv") is None:
        raise RuntimeError(
            "The tool uv is not available on this Colab machine, so the environment "
            "cannot be installed. Choose the runtime version "
            f'{_RUNTIME_VERSION} ("Runtime", "Change runtime type") and run this '
            "cell again."
        )

    setup_dir.mkdir(parents=True, exist_ok=True)
    overrides_file = setup_dir / _OVERRIDES_NAME
    overrides_file.write_text("\n".join(_OVERRIDES) + "\n", encoding="utf-8")
    print(_INSTALL_NOTICE, flush=True)
    started = _time.monotonic()
    script = setup_dir / _DOWNLOAD_SCRIPT_NAME
    try:
        # The overrides of pyproject.toml keep scipy and statsmodels at the versions of
        # the book; a plain install of the package does not read them.
        _run(
            [
                "uv",
                "pip",
                "install",
                "--system",
                "--override",
                str(overrides_file),
                f"fnirs-tutorials @ git+https://github.com/{_REPOSITORY}@{_REF}",
            ]
        )
        installed = _time.monotonic()
        # The package above comes through git, this script through raw.githubusercontent.com,
        # which answers with "cache-control: max-age=300". For up to five minutes after a
        # push to main, the script can be the earlier one while the package is the new one.
        _fetch(
            f"https://raw.githubusercontent.com/{_REPOSITORY}/{_REF}/scripts/download_data.py",
            script,
        )
        _run([_sys.executable, "-u", str(script), "--data-dir", _DATA_DIR])
        downloaded = _time.monotonic()
        _check_installed()
    except Exception as error:
        # No marker is written, so running the cell again starts over. An interruption is
        # not caught here: it is not a failure of a step.
        raise RuntimeError(f"The setup failed. {error} {_ADVICE}") from error

    marker.write_text(
        _json.dumps({"ref": _REF, "overrides": _OVERRIDES}) + "\n", encoding="utf-8"
    )
    _set_environment()
    print(
        f"{_TIMES_PREFIX} install {installed - started:.1f}, "
        f"download {downloaded - installed:.1f}, total {downloaded - started:.1f}",
        flush=True,
    )
    _end_session()


if _on_colab():
    _setup()

In [ ]:
import cedalion
import cedalion.io
import cedalion.nirs
import matplotlib.pyplot as plt
import numpy as np

from fnirs_tutorials import paths, style

style.apply_style()

In [ ]:
# The path is built from the data directory of the project, never from an absolute
# path.
snirf_path = (
    paths.data_dir()
    / "fresh"
    / "FRESH-Motor"
    / "sub-01"
    / "ses-left2s"
    / "nirs"
    / "sub-01_ses-left2s_task-FRESHMOTOR_nirs.snirf"
)

# read_snirf returns a list of Recording objects, one per SNIRF data block; this file
# has one.
recording = cedalion.io.read_snirf(snirf_path)[0]

# 'amp' is the raw intensity (amplitude) time series, with dimensions (channel,
# wavelength, time). The file declares no unit for the measurement, so Cedalion
# attaches a dimensionless placeholder; dequantify() moves it to an attribute so that
# plain NumPy access below does not warn about a stripped unit.
amplitudes = recording.timeseries["amp"].pint.dequantify()

# Both numbers are read from the loaded recording, never typed in.
n_channels = amplitudes.sizes["channel"]
sampling_rate_hz = 1.0 / float(np.diff(amplitudes.time.values).mean())
print(f"Channels (source-detector pairs): {n_channels}")
print(f"Wavelengths (nm): {amplitudes.wavelength.values.tolist()}")
print(f"Sampling rate: {sampling_rate_hz:.2f} Hz")

In [ ]:
# Illustration constants. The pick below only serves this illustration: choosing the
# channel closest to the target distance as the long channel is a heuristic, not a
# methodological recommendation.
LONG_CHANNEL_TARGET_CM = 3.0  # source-detector distance sought for the long channel
WINDOW_S = 2.0  # duration of the plotted window, from the start of the recording

# channel_distances gives the source-detector distance of every channel, in the length
# unit of the probe geometry (metres in this file); .pint.to("cm") converts it.
distances_cm = (
    cedalion.nirs.channel_distances(amplitudes, recording.geo3d)
    .pint.to("cm")
    .pint.dequantify()
)

short_channel = str(distances_cm.idxmin("channel").item())
long_channel = str(
    np.abs(distances_cm - LONG_CHANNEL_TARGET_CM).idxmin("channel").item()
)
if short_channel == long_channel:
    raise ValueError(
        f"The shortest channel and the channel closest to {LONG_CHANNEL_TARGET_CM} cm "
        f"are the same channel ({short_channel}); this recording has no distinct pair "
        "to compare."
    )

window = amplitudes.sel(time=slice(0.0, WINDOW_S))

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 5))
for ax, channel in zip(axes, [short_channel, long_channel], strict=True):
    distance = float(distances_cm.sel(channel=channel))
    for wavelength in window.wavelength.values:
        trace = window.sel(channel=channel, wavelength=wavelength)
        ax.plot(
            trace.time.values,
            trace.values,
            label=f"{channel}, {wavelength:.0f} nm, {distance:.2f} cm",
        )
    ax.set_ylabel("Raw intensity (a.u.)")
    ax.legend(loc="upper right")
axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    "Raw intensity on the shortest channel and on the channel closest to "
    f"{LONG_CHANNEL_TARGET_CM:g} cm"
)
plt.show()

## Pitfall demonstration

This section is a placeholder that exercises the fourth part of the chapter template. A real chapter shows here, on data, one way the analysis goes wrong.

## In the field / in LIONirs

This box is a placeholder that exercises the fifth part of the chapter template.

```{note} In the field / in LIONirs
A real chapter says here how the same step is done in the field and in the LIONirs toolbox.
```

## Self-check and further reading

This section is a placeholder that exercises the sixth part of the chapter template. A real chapter ends with three to five questions and a short list of verified references.